# Publish: Supabase Upsert

**Purpose**: Upsert Gold and approved Audit tables from Unity Catalog to Supabase Postgres for consumer-facing publication.

**Key Features**:
- Deterministic upsert with conflict resolution
- Row count validation pre/post sync
- Checksum verification
- Batch processing with progress tracking
- Explicit table dependencies and load order
- Connection pooling and retry logic

**Target Tables**:
- Gold layer: `workspace.gold.*`
- Audit layer: Approved audit tables only

In [0]:
%pip install psycopg2-binary sqlalchemy pandas --quiet
dbutils.library.restartPython()

In [0]:
import os
import json
import hashlib
from datetime import datetime
from typing import List, Dict, Tuple
import psycopg2
from psycopg2.extras import execute_batch
from sqlalchemy import create_engine, text
import pandas as pd
from pyspark.sql import DataFrame

# Supabase Connection Configuration
# In production, use Databricks Secrets for credentials
SUPABASE_HOST = dbutils.secrets.get(scope="supabase", key="host")
SUPABASE_PORT = dbutils.secrets.get(scope="supabase", key="port")
SUPABASE_DB = dbutils.secrets.get(scope="supabase", key="database")
SUPABASE_USER = dbutils.secrets.get(scope="supabase", key="username")
SUPABASE_PASSWORD = dbutils.secrets.get(scope="supabase", key="password")

# Publication Configuration
CATALOG = "workspace"
WAREHOUSE_SCHEMA = "warehouse"  # Dimensions and Facts
GOLD_SCHEMA = "gold"  # Analytics marts
AUDIT_SCHEMA = "audit"  # Audit tables
BATCH_SIZE = 10000  # Rows per batch
MAX_RETRIES = 3

# Connection string
CONN_STRING = f"postgresql://{SUPABASE_USER}:{SUPABASE_PASSWORD}@{SUPABASE_HOST}:{SUPABASE_PORT}/{SUPABASE_DB}"
engine = create_engine(CONN_STRING, pool_size=10, max_overflow=20)

print(f"✓ Configuration loaded")
print(f"✓ Target: {SUPABASE_HOST}/{SUPABASE_DB}")

In [0]:
# Explicit load order with dependencies
# Format: (table_name, schema, primary_key, dependencies)

# Warehouse tables (dimensions and facts)
WAREHOUSE_TABLES = [
    # Dimensions (no dependencies)
    ("dim_source", "warehouse", "source_sk", []),
    ("dim_sector", "warehouse", "sector_sk", []),
    ("dim_role", "warehouse", "role_sk", []),
    ("dim_location", "warehouse", "location_sk", []),
    ("dim_company", "warehouse", "company_sk", []),
    ("dim_skill", "warehouse", "skill_sk", []),
    ("dim_job", "warehouse", "job_sk", []),
    ("dim_company_alias", "warehouse", None, ["dim_company"]),  # Composite key
    
    # Facts (depend on dimensions)
    ("fact_job_postings", "warehouse", "fact_job_posting_sk", 
     ["dim_source", "dim_sector", "dim_role", "dim_location", "dim_company", "dim_job"]),
    ("fact_job_lifecycle", "warehouse", "fact_job_lifecycle_sk", ["dim_job"]),
    ("fact_salary", "warehouse", "fact_salary_sk", ["dim_job", "dim_location"]),
    ("fact_pipeline_runs", "warehouse", "fact_pipeline_run_sk", []),
    
    # Bridge tables
    ("bridge_job_skill", "warehouse", None, ["dim_job", "dim_skill"]),  # Composite key
]

# Gold analytics tables (depend on warehouse)
GOLD_TABLES = [
    ("gold_hiring_trends", "gold", None, ["fact_job_postings"]),  # Composite key
    ("gold_sector_overview", "gold", None, ["fact_job_postings"]),
    ("gold_location_trends", "gold", None, ["fact_job_postings"]),
    ("gold_company_hiring", "gold", None, ["fact_job_postings", "dim_company"]),
    ("gold_salary_trends", "gold", None, ["fact_salary"]),
    ("gold_skill_demand", "gold", None, ["bridge_job_skill"]),
    ("gold_hospitality_companies", "gold", None, ["dim_company"]),
    ("gold_hospitality_hiring", "gold", None, ["fact_job_postings"]),
    ("gold_hospitality_skills", "gold", None, ["bridge_job_skill"]),
    ("gold_pipeline_health", "gold", None, ["fact_pipeline_runs"]),
]

# Audit tables (independent)
AUDIT_TABLES = [
    ("audit_pipeline_runs", "audit", None, []),
    ("audit_dq_results", "audit", None, []),
    ("audit_access_events", "audit", None, []),
]

# Combine all tables in load order
ALL_TABLES = WAREHOUSE_TABLES + GOLD_TABLES + AUDIT_TABLES

print(f"✓ Load order defined:")
print(f"  Warehouse: {len(WAREHOUSE_TABLES)} tables (dimensions, facts, bridges)")
print(f"  Gold: {len(GOLD_TABLES)} tables (analytics marts)")
print(f"  Audit: {len(AUDIT_TABLES)} tables")
print(f"  Total: {len(ALL_TABLES)} tables")

In [0]:
def calculate_checksum(df: pd.DataFrame) -> str:
    """Calculate MD5 checksum of DataFrame content."""
    content_str = df.to_csv(index=False, header=True)
    return hashlib.md5(content_str.encode()).hexdigest()

def get_table_row_count(table_full_name: str) -> int:
    """Get row count from Unity Catalog table."""
    count = spark.sql(f"SELECT COUNT(*) as cnt FROM {table_full_name}").collect()[0]['cnt']
    return count

def get_postgres_row_count(table_name: str) -> int:
    """Get row count from Postgres table."""
    with engine.connect() as conn:
        result = conn.execute(text(f"SELECT COUNT(*) as cnt FROM {table_name}"))
        return result.fetchone()[0]

def create_postgres_table_if_not_exists(table_name: str, df: pd.DataFrame, primary_key: str = None):
    """Create Postgres table from DataFrame schema if it doesn't exist."""
    with engine.connect() as conn:
        # Check if table exists
        result = conn.execute(text(
            f"SELECT EXISTS (SELECT FROM information_schema.tables WHERE table_name = '{table_name}')"
        ))
        exists = result.fetchone()[0]
        
        if not exists:
            # Create table from pandas DataFrame
            df.head(0).to_sql(table_name, engine, if_exists='replace', index=False)
            
            # Add primary key constraint if specified
            if primary_key:
                conn.execute(text(f"ALTER TABLE {table_name} ADD PRIMARY KEY ({primary_key})"))
                conn.commit()
            
            print(f"  ✓ Created table: {table_name}")
        else:
            print(f"  ✓ Table exists: {table_name}")

def upsert_batch(table_name: str, df: pd.DataFrame, primary_key: str = None) -> Dict:
    """Upsert batch of data with conflict resolution."""
    start_time = datetime.now()
    
    if df.empty:
        return {"rows_processed": 0, "duration_sec": 0}
    
    # Create table if needed
    create_postgres_table_if_not_exists(table_name, df, primary_key)
    
    # Prepare upsert query
    columns = list(df.columns)
    col_names = ", ".join(columns)
    placeholders = ", ".join(["%s"] * len(columns))
    
    if primary_key and primary_key in columns:
        # Use ON CONFLICT for upsert
        update_cols = ", ".join([f"{col} = EXCLUDED.{col}" for col in columns if col != primary_key])
        query = f"""
            INSERT INTO {table_name} ({col_names})
            VALUES ({placeholders})
            ON CONFLICT ({primary_key}) DO UPDATE SET {update_cols}
        """
    else:
        # Insert only (for tables without primary key)
        query = f"INSERT INTO {table_name} ({col_names}) VALUES ({placeholders})"
    
    # Execute batch
    conn = psycopg2.connect(
        host=SUPABASE_HOST,
        port=SUPABASE_PORT,
        database=SUPABASE_DB,
        user=SUPABASE_USER,
        password=SUPABASE_PASSWORD
    )
    
    try:
        with conn.cursor() as cursor:
            data = [tuple(row) for row in df.values]
            execute_batch(cursor, query, data, page_size=1000)
            conn.commit()
            rows_processed = len(data)
    except Exception as e:
        conn.rollback()
        raise e
    finally:
        conn.close()
    
    duration = (datetime.now() - start_time).total_seconds()
    
    return {
        "rows_processed": rows_processed,
        "duration_sec": round(duration, 2)
    }

print("✓ Utility functions loaded")

In [0]:
def upsert_table_to_supabase(table_name: str, primary_key: str = None, schema: str = GOLD_SCHEMA) -> Dict:
    """
    Upsert entire table from Unity Catalog to Supabase.
    Returns metadata about the sync operation.
    """
    print(f"\n{'='*60}")
    print(f"Syncing: {CATALOG}.{schema}.{table_name}")
    print(f"{'='*60}")
    
    start_time = datetime.now()
    table_full_name = f"{CATALOG}.{schema}.{table_name}"
    
    # 1. Get source row count
    source_count = get_table_row_count(table_full_name)
    print(f"Source rows: {source_count:,}")
    
    if source_count == 0:
        print("⚠️  Source table is empty, skipping...")
        return {
            "table": table_name,
            "status": "skipped",
            "reason": "empty_source"
        }
    
    # 2. Load source data
    print("Loading source data...")
    spark_df = spark.table(table_full_name)
    pandas_df = spark_df.toPandas()
    
    # 3. Calculate checksum
    checksum = calculate_checksum(pandas_df)
    print(f"Checksum: {checksum}")
    
    # 4. Upsert in batches
    total_rows = len(pandas_df)
    batches = (total_rows // BATCH_SIZE) + 1
    print(f"Upserting in {batches} batch(es)...")
    
    total_processed = 0
    for i in range(batches):
        start_idx = i * BATCH_SIZE
        end_idx = min((i + 1) * BATCH_SIZE, total_rows)
        batch_df = pandas_df.iloc[start_idx:end_idx]
        
        result = upsert_batch(table_name, batch_df, primary_key)
        total_processed += result['rows_processed']
        
        print(f"  Batch {i+1}/{batches}: {result['rows_processed']:,} rows in {result['duration_sec']}s")
    
    # 5. Validate target row count
    target_count = get_postgres_row_count(table_name)
    print(f"\nTarget rows: {target_count:,}")
    
    # 6. Validate count matches
    if source_count != target_count:
        print(f"⚠️  WARNING: Row count mismatch! Source={source_count}, Target={target_count}")
        status = "warning"
    else:
        print(f"✓ Row count validated")
        status = "success"
    
    duration = (datetime.now() - start_time).total_seconds()
    print(f"\n✓ Completed in {duration:.2f}s")
    
    return {
        "table": table_name,
        "status": status,
        "source_rows": source_count,
        "target_rows": target_count,
        "checksum": checksum,
        "duration_sec": round(duration, 2),
        "synced_at": datetime.now().isoformat()
    }

print("✓ Main upsert function loaded")

In [0]:
# Execute sync for all tables in load order
results = []

print("\n" + "="*80)
print("STARTING FULL SYNC TO SUPABASE")
print("="*80)

# Sync all tables in dependency order
for table_name, schema, primary_key, dependencies in ALL_TABLES:
    try:
        result = upsert_table_to_supabase(table_name, primary_key, schema=schema)
        results.append(result)
    except Exception as e:
        print(f"\n❌ ERROR syncing {schema}.{table_name}: {str(e)}")
        results.append({
            "table": f"{schema}.{table_name}",
            "status": "error",
            "error": str(e)
        })
        # Continue with next table
        continue

print("\n" + "="*80)
print("SYNC COMPLETE")
print("="*80)

In [0]:
# Generate summary report
import pandas as pd

report_df = pd.DataFrame(results)
print("\n" + "="*80)
print("SYNC SUMMARY REPORT")
print("="*80)
print(report_df.to_string(index=False))

# Status breakdown
print("\n" + "="*80)
print("STATUS BREAKDOWN")
print("="*80)
status_counts = report_df['status'].value_counts()
for status, count in status_counts.items():
    print(f"{status.upper()}: {count} tables")

# Total metrics
if 'source_rows' in report_df.columns:
    total_source = report_df['source_rows'].sum()
    total_target = report_df['target_rows'].sum()
    total_duration = report_df['duration_sec'].sum()
    
    print("\n" + "="*80)
    print("TOTALS")
    print("="*80)
    print(f"Total source rows: {total_source:,}")
    print(f"Total target rows: {total_target:,}")
    print(f"Total duration: {total_duration:.2f}s ({total_duration/60:.2f} min)")
    
    if total_source == total_target:
        print("\n✓ All row counts validated successfully!")
    else:
        print(f"\n⚠️  Row count discrepancy: {abs(total_source - total_target):,} rows")

# Save results to Unity Catalog for audit trail
result_spark_df = spark.createDataFrame(report_df)
result_spark_df.write.mode("append").saveAsTable(f"{CATALOG}.{AUDIT_SCHEMA}.publish_sync_log")

print("\n✓ Sync results logged to audit.publish_sync_log")